# memory-efficient geometric summary analysis

This notebook is written to avoid filling RAM.

The main idea is:

1. **Never reconstruct the full session activity matrix.**
   Instead, for each session and area, reconstruct **only the selected neurons** from the stored SVD factors.

2. **Process one `(dataset, area, alignment)` block at a time.**
   Compute metrics immediately, save them to disk, then free memory.

3. **Reload only the saved metric tables** for the group summaries and plots.

The first two metrics are computed on **speed-residualized non-z-scored activity**.
All other metrics are computed on **speed-residualized z-scored activity**.

In [28]:
import os
import gc
import json
import math
import pickle
from pathlib import Path
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from scipy.linalg import subspace_angles
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [29]:
# ---------------------------------------------------------------------
# paths
# ---------------------------------------------------------------------
base = Path.cwd()
for p in [base, *base.parents]:
    if (p / "data").exists():
        project_root = p
        break
else:
    project_root = base

data_root = project_root / "data"

analysis_root = project_root / "results" / "geometric_summary_streaming"
session_metrics_dir = analysis_root / "session_metrics"
session_rdms_dir = analysis_root / "session_rdms"
summary_dir = analysis_root / "summary"
plot_dir = analysis_root / "plots"

for d in [analysis_root, session_metrics_dir, session_rdms_dir, summary_dir, plot_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("project_root =", project_root)
print("analysis_root =", analysis_root)

project_root = /Users/johnmadrid/GitHub/isp-unsupervised-learning
analysis_root = /Users/johnmadrid/GitHub/isp-unsupervised-learning/results/geometric_summary_streaming


In [30]:
# ---------------------------------------------------------------------
# dataset configuration
# edit this block to match the sessions you want to process
# ---------------------------------------------------------------------

MAX_SAMPLE = {'V1': 12000, "mHV":6000, "lHV":3000, "aHV":3000}
# # cohort 1
sup_bef   = "VR2_2021_03_20_1"
sup_aft   = "VR2_2021_04_06_1"
unsup_bef = "TX105_2022_10_08_2"
unsup_aft = "TX105_2022_10_19_2"

# # cohort 2
# sup_bef   = "TX60_2021_04_10_1"
# sup_aft   = "TX60_2021_05_04_1"
# unsup_bef = "TX123_2023_12_21_1"
# unsup_aft = "TX123_2024_01_02_1"

# cohort 3
# sup_bef   = "TX108_2023_03_13_1"
# sup_aft   = "TX108_2023_03_22_1"
# unsup_bef = "TX83_2022_08_17_1"
# unsup_aft = "TX83_2022_08_29_1"

TARGETS = [
    (sup_bef,   "sup_bef",   "rewarded",   "day1"),
    (sup_aft,   "sup_aft",   "rewarded",   "day14"),
    (unsup_bef, "unsup_bef", "unrewarded", "day1"),
    (unsup_aft, "unsup_aft", "unrewarded", "day14"),
]

AREAS = ["V1", "mHV", "lHV", "aHV"]

BUFFER_TUNNEL = (2, 8)
BUFFER_SOUND  = (5, 5)

ALIGNMENTS = {
    "Trial_start_time": BUFFER_TUNNEL,
    "SoundTime": BUFFER_SOUND,
}

# epoch windows are inclusive frame ranges in aligned coordinates
# example: with Trial_start_time buffer (2, 8), frames are [-2, -1, 0, ..., 8]
# edit these as needed
EPOCH_WINDOWS = {
    "Trial_start_time": {
        "entrance": (-1, 2),
        "post_entrance": (3, 6),
    },
    "SoundTime": {
        "cue": (-1, 2),
        "late_cue": (3, 5),
    },
}

# which stimulus labels define the main stimulus contrast
STIM_PAIR = ("leaf1", "circle1")

# optional behavior keys for outcome / reward labels, tried in order
# if none exist, the stimulus-reward-axis angle is skipped
REWARD_LABEL_CANDIDATES = [
    "Rewarded",
    "rewarded",
    "isRewarded",
    "reward",
    "Outcome",
    "TrialOutcome",
]

# ridge-style covariance regularization for Mahalanobis
MAHAL_REG = 1e-6

# top-k dimension for principal-angle calculation
PRINCIPAL_ANGLE_K = 5

# decoding CV
DECODER_N_SPLITS = 5
DECODER_RANDOM_STATE = 0


RDM_CONDITIONS = [
    ("leaf1",   "entrance", "early"),
    ("leaf1",   "entrance", "late"),
    ("circle1", "entrance", "early"),
    ("circle1", "entrance", "late"),
    ("leaf1",   "cue",      "early"),
    ("leaf1",   "cue",      "late"),
    ("circle1", "cue",      "early"),
    ("circle1", "cue",      "late"),
]


PR_SUBSAMPLE_GRID = [1000, 2000, 3000, 4000, 5000, 6000,
                     7000, 8000, 9000, 10000, 11000, 12000]

PR_SUBSAMPLE_N_REPEATS = 20
PR_SUBSAMPLE_RANDOM_SEED = 0
PR_SCALING_USE = "z"   


pr_scaling_dir = analysis_root / "pr_scaling"

for d in [analysis_root, session_metrics_dir, session_rdms_dir, summary_dir, plot_dir, pr_scaling_dir]:
    d.mkdir(parents=True, exist_ok=True)

In [31]:
# ---------------------------------------------------------------------
# file helpers
# ---------------------------------------------------------------------
def get_beh_path(target_file):
    mapping = {
        "VR2_2021_03_20_1": "Beh_sup_train1_before_learning.npy",
        "VR2_2021_04_06_1": "Beh_sup_train1_after_learning.npy",
        "TX105_2022_10_08_2": "Beh_unsup_train1_before_learning.npy",
        "TX105_2022_10_19_2": "Beh_unsup_train1_after_learning.npy",
        "TX60_2021_04_10_1": "Beh_sup_train1_before_learning.npy",
        "TX60_2021_05_04_1": "Beh_sup_train1_after_learning.npy",
        "TX123_2023_12_21_1": "Beh_unsup_train1_before_learning.npy",
        "TX123_2024_01_02_1": "Beh_unsup_train1_after_learning.npy",
        "TX108_2023_03_13_1": "Beh_sup_train1_before_learning.npy",
        "TX108_2023_03_22_1": "Beh_sup_train1_after_learning.npy",
        "TX83_2022_08_17_1": "Beh_unsup_train1_before_learning.npy",
        "TX83_2022_08_29_1": "Beh_unsup_train1_after_learning.npy",
    }
    if target_file not in mapping:
        raise KeyError(f"no behavior mapping for {target_file}")
    return data_root / mapping[target_file]

def load_behavior_dict(target_file):
    beh_path = get_beh_path(target_file)
    beh_all = np.load(beh_path, allow_pickle=True).item()
    return beh_all[target_file]

def load_neuron_area_series(recording_name):
    filename = data_root / f"{recording_name[:-1]}trans.npz"
    with np.load(filename, allow_pickle=True) as retin:
        iarea = retin["iarea"]
    area_map = {
        8: "V1",
        0: "mHV", 1: "mHV", 2: "mHV", 9: "mHV",
        5: "lHV", 6: "lHV",
        3: "aHV", 4: "aHV",
    }
    areas = pd.Series([area_map.get(int(x), "Other") for x in iarea], name="area")
    areas.index = np.arange(len(areas), dtype=int)
    return areas

def get_neuron_indices_by_area(recording_name, area_label):
    area_series = load_neuron_area_series(recording_name)
    return area_series.index[area_series.eq(area_label)].to_numpy(dtype=int)

def load_selected_activity_from_svd(target_file, neuron_ids):
    '''
    Memory-efficient reconstruction.

    Original code reconstructed the full matrix:
        spk = (svd["U"].T @ svd["V"]).T

    Here we only reconstruct the selected neurons:
        spk_sel = (U[:, neuron_ids].T @ V).T

    so the returned array has shape (n_time, n_selected_neurons).
    '''
    spike_path = data_root / f"{target_file}_SVD_dec.npy"
    svd = np.load(spike_path, allow_pickle=True).item()
    U = np.asarray(svd["U"])
    V = np.asarray(svd["V"])
    spk_sel = (U[:, neuron_ids].T @ V).T.astype(np.float32, copy=False)
    del svd, U, V
    gc.collect()
    return spk_sel



RDM_CONDITIONS = [
    ("leaf1",   "entrance", "early"),
    ("leaf1",   "entrance", "late"),
    ("circle1", "entrance", "early"),
    ("circle1", "entrance", "late"),
    ("leaf1",   "cue",      "early"),
    ("leaf1",   "cue",      "late"),
    ("circle1", "cue",      "early"),
    ("circle1", "cue",      "late"),
]

In [32]:
# ---------------------------------------------------------------------
# tensor construction and preprocessing
# ---------------------------------------------------------------------
def build_tensor(trial_timestamps, buffer, spiking_data, spiking_timestamps):
    n_trials = len(trial_timestamps)
    n_time = buffer[0] + buffer[1] + 1
    n_neurons = spiking_data.shape[1]
    tensor = np.zeros((n_neurons, n_time, n_trials), dtype=np.float32)
    for tr, timestamp in enumerate(trial_timestamps):
        idx = np.searchsorted(spiking_timestamps, timestamp, side="right")
        tensor[:, :, tr] = spiking_data[idx - buffer[0]: idx + buffer[1] + 1, :].T
    return tensor

def build_speed_matrix(trial_timestamps, buffer, spiking_timestamps, run_speed):
    n_time = buffer[0] + buffer[1] + 1
    idx = np.searchsorted(spiking_timestamps, trial_timestamps, side="right")
    return run_speed[np.arange(n_time)[:, None] + (idx - buffer[0])].astype(np.float32)

def speed_correct(tensor, speed_matrix):
    n_neurons, n_time, n_trials = tensor.shape
    n_obs = n_time * n_trials
    X = np.c_[np.ones(n_obs, dtype=np.float64), speed_matrix.ravel().astype(np.float64)]
    Y = tensor.reshape(n_neurons, n_obs).T.astype(np.float64)
    B = np.linalg.lstsq(X, Y, rcond=None)[0]
    resid = (Y - X @ B).T.reshape(n_neurons, n_time, n_trials).astype(np.float32)
    return resid

def robust_zscore_tensor(tensor):
    flat = tensor.reshape(tensor.shape[0], -1)
    med = np.median(flat, axis=1, keepdims=True)
    mad = 1.4826 * np.median(np.abs(flat - med), axis=1, keepdims=True)
    mad = np.where(mad > 1e-12, mad, 1e-12)
    z = ((flat - med) / mad).reshape(tensor.shape).astype(np.float32)
    return z

def frame_axis_from_buffer(buffer):
    return np.arange(-buffer[0], buffer[1] + 1, dtype=int)

def epoch_mask_from_window(frame_axis, frame_window):
    lo, hi = frame_window
    return (frame_axis >= lo) & (frame_axis <= hi)

def epoch_trial_matrix(tensor, frame_mask):
    '''
    tensor: (n_neurons, n_time, n_trials)
    return X: (n_trials, n_neurons), formed by averaging activity over the epoch window
    '''
    return tensor[:, frame_mask, :].mean(axis=1).T.astype(np.float32)

In [33]:
# ---------------------------------------------------------------------
# metric helpers
# ---------------------------------------------------------------------

def participation_ratio_subsampled( X, neuron_grid, n_repeats=20, random_seed=0,):
    """
    X: (n_trials, n_neurons)

    For each neuron count in neuron_grid:
    - if X has fewer neurons than requested, skip
    - otherwise subsample without replacement n_repeats times
    - compute PR on each subsample

    Returns a long-format DataFrame.
    """
    rng = np.random.default_rng(random_seed)
    n_trials, n_neurons = X.shape

    rows = []
    for n_sub in neuron_grid:
        print('N_sub: ', n_sub)
        if n_sub > n_neurons:
            continue

        for rep in range(n_repeats):
            neuron_idx = rng.choice(n_neurons, size=n_sub, replace=False)
            X_sub = X[:, neuron_idx]
            pr = participation_ratio(X_sub)

            rows.append({ "n_neurons_subsample": int(n_sub),  "repeat": int(rep),
                "participation_ratio": float(pr) if pr is not None else np.nan, })

    return pd.DataFrame(rows)

def safe_vector_norm(x):
    x = np.asarray(x, dtype=float)
    return float(np.sqrt(np.sum(x * x)))

def euclidean_distance_between_means(X_a, X_b):
    mean_a = X_a.mean(axis=0)
    mean_b = X_b.mean(axis=0)
    return safe_vector_norm(mean_a - mean_b)

def mahalanobis_distance_between_means(X_a, X_b, reg=1e-6):
    mean_a = X_a.mean(axis=0)
    mean_b = X_b.mean(axis=0)
    cov_a = np.cov(X_a, rowvar=False)
    cov_b = np.cov(X_b, rowvar=False)
    cov_pooled = 0.5 * (np.atleast_2d(cov_a) + np.atleast_2d(cov_b))
    cov_pooled = cov_pooled + reg * np.eye(cov_pooled.shape[0])
    diff = mean_a - mean_b
    try:
        inv_cov = np.linalg.inv(cov_pooled)
    except np.linalg.LinAlgError:
        inv_cov = np.linalg.pinv(cov_pooled)
    val = float(diff @ inv_cov @ diff)
    return math.sqrt(max(val, 0.0))

def decoding_accuracy_linear_svm(X, y, n_splits=5, random_state=0):
    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    if len(classes) < 2:
        return np.nan
    max_splits = counts.min()
    n_splits = int(min(n_splits, max_splits))
    if n_splits < 2:
        return np.nan
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    clf = make_pipeline(StandardScaler(with_mean=True, with_std=True), LinearSVC(dual=False, max_iter=5000))
    return float(cross_val_score(clf, X, y, cv=cv).mean())

def participation_ratio(X):
    Xc = X - X.mean(axis=0, keepdims=True)
    if Xc.shape[0] < 2:
        return np.nan
    cov = (Xc.T @ Xc) / (Xc.shape[0] - 1)
    eigs = np.linalg.eigvalsh(cov)
    eigs = eigs[eigs > 1e-12]
    if eigs.size == 0:
        return np.nan
    return float((eigs.sum() ** 2) / np.sum(eigs ** 2))

def principal_angle_mean(X_a, X_b, k=5):
    Xa = X_a - X_a.mean(axis=0, keepdims=True)
    Xb = X_b - X_b.mean(axis=0, keepdims=True)
    Ua, Sa, Vta = np.linalg.svd(Xa, full_matrices=False)
    Ub, Sb, Vtb = np.linalg.svd(Xb, full_matrices=False)
    ka = min(k, Vta.shape[0])
    kb = min(k, Vtb.shape[0])
    kk = min(ka, kb)
    if kk < 1:
        return np.nan
    basis_a = Vta[:kk].T
    basis_b = Vtb[:kk].T
    ang = subspace_angles(basis_a, basis_b)
    return float(np.mean(ang))

def mean_noise_correlation(X_a, X_b):
    resid_a = X_a - X_a.mean(axis=0, keepdims=True)
    resid_b = X_b - X_b.mean(axis=0, keepdims=True)
    resid = np.vstack([resid_a, resid_b])
    if resid.shape[1] < 2:
        return np.nan
    corr = np.corrcoef(resid, rowvar=False)
    iu = np.triu_indices_from(corr, k=1)
    vals = corr[iu]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan
    return float(vals.mean())

def signal_noise_angle(X_a, X_b):
    resid_a = X_a - X_a.mean(axis=0, keepdims=True)
    resid_b = X_b - X_b.mean(axis=0, keepdims=True)
    resid = np.vstack([resid_a, resid_b])

    signal = X_a.mean(axis=0) - X_b.mean(axis=0)
    signal_norm = safe_vector_norm(signal)
    if signal_norm < 1e-12:
        return np.nan
    signal = signal / signal_norm

    Rc = resid - resid.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Rc, full_matrices=False)
    noise_axis = Vt[0]
    noise_norm = safe_vector_norm(noise_axis)
    if noise_norm < 1e-12:
        return np.nan
    noise_axis = noise_axis / noise_norm

    cosang = np.clip(np.abs(np.dot(signal, noise_axis)), 0.0, 1.0)
    return float(np.arccos(cosang))

def angle_between_axes(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na = safe_vector_norm(a)
    nb = safe_vector_norm(b)
    if na < 1e-12 or nb < 1e-12:
        return np.nan
    cosang = np.clip(np.abs(np.dot(a / na, b / nb)), 0.0, 1.0)
    return float(np.arccos(cosang))

def build_rdm_from_centroids(centroid_matrix):
    '''
    centroid_matrix: (n_conditions, n_features)
    returns Euclidean RDM: (n_conditions, n_conditions)
    '''
    X = np.asarray(centroid_matrix, dtype=float)
    sq = np.sum(X ** 2, axis=1, keepdims=True)
    dist2 = np.maximum(sq + sq.T - 2 * (X @ X.T), 0.0)
    return np.sqrt(dist2)

def upper_triangle_values(M):
    iu = np.triu_indices_from(M, k=1)
    return M[iu]

def build_combined_rdm_for_area( beh,spk,ft, run_speed,
    stim_pair=("leaf1", "circle1"), n_early_late=100,):
    
    # --- build tensors for both alignments
    trial_ts_entrance = np.asarray(beh["Trial_start_time"])
    trial_ts_cue = np.asarray(beh["SoundTime"])

    raw_tensor_entrance = build_tensor(trial_ts_entrance, ALIGNMENTS["Trial_start_time"], spk, ft)
    raw_tensor_cue = build_tensor(trial_ts_cue, ALIGNMENTS["SoundTime"], spk, ft)

    speed_entrance = build_speed_matrix(trial_ts_entrance, ALIGNMENTS["Trial_start_time"], ft, run_speed)
    speed_cue = build_speed_matrix(trial_ts_cue, ALIGNMENTS["SoundTime"], ft, run_speed)

    resid_entrance = speed_correct(raw_tensor_entrance, speed_entrance)
    resid_cue = speed_correct(raw_tensor_cue, speed_cue)

    z_entrance = robust_zscore_tensor(resid_entrance)
    z_cue = robust_zscore_tensor(resid_cue)

    frame_axis_entrance = frame_axis_from_buffer(ALIGNMENTS["Trial_start_time"])
    frame_axis_cue = frame_axis_from_buffer(ALIGNMENTS["SoundTime"])

    entrance_mask = epoch_mask_from_window(frame_axis_entrance, EPOCH_WINDOWS["Trial_start_time"]["entrance"] )
    cue_mask = epoch_mask_from_window(frame_axis_cue, EPOCH_WINDOWS["SoundTime"]["cue"])

    X_entrance = epoch_trial_matrix(z_entrance, entrance_mask)   # (n_trials, n_neurons)
    X_cue = epoch_trial_matrix(z_cue, cue_mask)

    stim_labels = np.asarray(beh["TrialStim"][:X_entrance.shape[0]]).astype(str)

    early_mask, late_mask = get_early_late_masks(len(stim_labels), n_select=n_early_late)

    condition_defs = [
        ("leaf1",   "entrance", "early"),
        ("leaf1",   "entrance", "late"),
        ("circle1", "entrance", "early"),
        ("circle1", "entrance", "late"),
        ("leaf1",   "cue",      "early"),
        ("leaf1",   "cue",      "late"),
        ("circle1", "cue",      "early"),
        ("circle1", "cue",      "late"),
    ]

    centroid_vectors = []
    centroid_meta = []

    for stim, epoch, phase in condition_defs:
        if epoch == "entrance":
            X = X_entrance
        elif epoch == "cue":
            X = X_cue
        else:
            raise ValueError(f"Unknown epoch: {epoch}")

        if phase == "early":
            phase_mask = early_mask
        elif phase == "late":
            phase_mask = late_mask
        else:
            raise ValueError(f"Unknown phase: {phase}")

        cond_mask = (stim_labels == stim) & phase_mask
        X_cond = X[cond_mask]

        if X_cond.shape[0] == 0:
            return None

        centroid_vectors.append(X_cond.mean(axis=0).astype(np.float32))
        centroid_meta.append({
            "condition": f"{stim}_{epoch}_{phase}",
            "stimulus": stim,
            "epoch": epoch,
            "phase": phase,
            "n_trials": int(X_cond.shape[0]),
        })

    centroid_mat = np.vstack(centroid_vectors)
    rdm = build_rdm_from_centroids(centroid_mat)

    return {
        "meta": centroid_meta,
        "rdm": rdm.astype(np.float32),
    }

In [34]:
# ---------------------------------------------------------------------
# behavior label helpers
# ---------------------------------------------------------------------
def get_trial_labels(beh, n_trials):
    stim = np.asarray(beh["TrialStim"][:n_trials]).astype(str)
    trind = np.asarray(beh["trInd"][:n_trials]).astype(int)
    return stim, trind

def infer_reward_labels(beh, n_trials):
    for key in REWARD_LABEL_CANDIDATES:
        if key in beh:
            vals = np.asarray(beh[key][:n_trials])
            if vals.ndim == 0:
                continue
            if vals.dtype.kind in {"b", "i", "u", "f"}:
                uniq = np.unique(vals[~pd.isna(vals)])
                if uniq.size >= 2:
                    return vals, key
            else:
                vals = vals.astype(str)
                uniq = np.unique(vals)
                if uniq.size >= 2:
                    return vals, key
    return None, None

def trial_subset_mask_for_stim_pair(stim_labels, stim_pair):
    return np.isin(stim_labels, list(stim_pair))

def split_two_conditions(X, labels, cond_a, cond_b):
    Xa = X[np.asarray(labels) == cond_a]
    Xb = X[np.asarray(labels) == cond_b]
    return Xa, Xb

## pass 1 — stream one session and one area at a time

This is the memory-safe pass.

For each `(dataset, area)`:
- reconstruct only the neurons from that area
- process one alignment at a time
- compute metrics immediately
- save metric rows to disk
- save the session-area RDM inputs needed later for cross-region comparisons
- free memory

In [35]:
# ---------------------------------------------------------------------
# pass 1
# ---------------------------------------------------------------------
def process_single_session_area(target_file, dataset_label, group_label, day_label, area):
    print(f"processing: {dataset_label} | {target_file} | {group_label} | {day_label} | {area}")

    neuron_ids = get_neuron_indices_by_area(target_file, area)
    if len(neuron_ids) == 0:
        print("  no neurons in this area; skipping")
        return None

    beh = load_behavior_dict(target_file)
    spk = load_selected_activity_from_svd(target_file, neuron_ids)

    ft = np.asarray(beh["ft"][: spk.shape[0] + 1])
    run_speed = np.asarray(beh["ft_RunSpeed"][: len(ft)], dtype=np.float32)

    all_metric_rows = []
    all_pr_scaling_rows = []
    

    for alignment_name, buffer_used in ALIGNMENTS.items():
        trial_timestamps = np.asarray(beh[alignment_name])
        raw_tensor = build_tensor(trial_timestamps, buffer_used, spk, ft)
        speed_matrix = build_speed_matrix(trial_timestamps, buffer_used, ft, run_speed)
        resid_tensor = speed_correct(raw_tensor, speed_matrix)
        z_tensor = robust_zscore_tensor(resid_tensor)

        frame_axis = frame_axis_from_buffer(buffer_used)
        stim_labels, trial_index = get_trial_labels(beh, raw_tensor.shape[2])
        reward_labels, reward_key = infer_reward_labels(beh, raw_tensor.shape[2])

        for epoch_name, frame_window in EPOCH_WINDOWS[alignment_name].items():
            fmask = epoch_mask_from_window(frame_axis, frame_window)
            if fmask.sum() == 0:
                print(f"  warning: empty epoch mask for {alignment_name} / {epoch_name}")
                continue

            X_resid = epoch_trial_matrix(resid_tensor, fmask)
            X_z = epoch_trial_matrix(z_tensor, fmask)

            if PR_SCALING_USE == "z":
                X_pr = X_z
            elif PR_SCALING_USE == "resid":
                X_pr = X_resid
            else:
                raise ValueError(f"Unknown PR_SCALING_USE: {PR_SCALING_USE}")
            
            pr_df = participation_ratio_subsampled(
                X_pr,
                neuron_grid=PR_SUBSAMPLE_GRID,
                n_repeats=PR_SUBSAMPLE_N_REPEATS,
                random_seed=PR_SUBSAMPLE_RANDOM_SEED,
            )
            
            if not pr_df.empty:
                pr_df["dataset_id"] = target_file
                pr_df["dataset_label"] = dataset_label
                pr_df["group"] = group_label
                pr_df["day"] = day_label
                pr_df["area"] = area
                pr_df["alignment"] = alignment_name
                pr_df["epoch"] = epoch_name
                pr_df["n_neurons_available"] = int(X_pr.shape[1])
                pr_df["n_trials"] = int(X_pr.shape[0])
                pr_df["pr_input_type"] = PR_SCALING_USE
            
                all_pr_scaling_rows.append(pr_df)
            if False:
                main_mask = trial_subset_mask_for_stim_pair(stim_labels, STIM_PAIR)
                X_resid_main = X_resid[main_mask]
                X_z_main = X_z[main_mask]
                y_main = stim_labels[main_mask]
    
                if np.sum(y_main == STIM_PAIR[0]) < 2 or np.sum(y_main == STIM_PAIR[1]) < 2:
                    print(f"  skipping main metrics for {alignment_name} / {epoch_name}: too few trials")
                else:
                    Xa_resid, Xb_resid = split_two_conditions(X_resid_main, y_main, STIM_PAIR[0], STIM_PAIR[1])
                    Xa_z, Xb_z = split_two_conditions(X_z_main, y_main, STIM_PAIR[0], STIM_PAIR[1])
    
                    metric_row = {
                        "dataset_id": target_file,
                        "dataset_label": dataset_label,
                        "group": group_label,
                        "day": day_label,
                        "area": area,
                        "alignment": alignment_name,
                        "epoch": epoch_name,
                        "n_neurons": int(X_z.shape[1]),
                        "n_trials_total": int(X_z.shape[0]),
                        "n_trials_main": int(X_z_main.shape[0]),
                        "n_leaf": int(Xa_z.shape[0]),
                        "n_circle": int(Xb_z.shape[0]),
                        "euclidean_distance_resid": euclidean_distance_between_means(Xa_resid, Xb_resid),
                        "mahalanobis_distance_resid": mahalanobis_distance_between_means(Xa_resid, Xb_resid, reg=MAHAL_REG),
                        "decoding_accuracy_z": decoding_accuracy_linear_svm(X_z_main, y_main, n_splits=DECODER_N_SPLITS, random_state=DECODER_RANDOM_STATE),
                        "participation_ratio_z": participation_ratio(X_z_main),
                        "principal_angle_mean_z": principal_angle_mean(Xa_z, Xb_z, k=PRINCIPAL_ANGLE_K),
                        "mean_noise_correlation_z": mean_noise_correlation(Xa_z, Xb_z),
                        "signal_noise_angle_z": signal_noise_angle(Xa_z, Xb_z),
                    }
    
                    if reward_labels is not None:
                        reward_epoch = reward_labels[fmask.any() and slice(None)]
                        reward_epoch = np.asarray(reward_labels[:X_z.shape[0]])
                        valid_reward = ~pd.isna(reward_epoch)
                        reward_epoch = reward_epoch[valid_reward]
                        X_reward = X_z[valid_reward]
                        if len(np.unique(reward_epoch)) >= 2:
                            uniq = list(pd.unique(reward_epoch))
                            if len(uniq) >= 2:
                                y0, y1 = uniq[0], uniq[1]
                                Xr0 = X_reward[np.asarray(reward_epoch) == y0]
                                Xr1 = X_reward[np.asarray(reward_epoch) == y1]
                                if Xr0.shape[0] >= 2 and Xr1.shape[0] >= 2:
                                    stim_axis = Xa_z.mean(axis=0) - Xb_z.mean(axis=0)
                                    reward_axis = Xr0.mean(axis=0) - Xr1.mean(axis=0)
                                    metric_row["stim_reward_axis_angle_z"] = angle_between_axes(stim_axis, reward_axis)
                                    metric_row["reward_label_key"] = reward_key
                                    metric_row["reward_label_a"] = str(y0)
                                    metric_row["reward_label_b"] = str(y1)
                                else:
                                    metric_row["stim_reward_axis_angle_z"] = np.nan
                            else:
                                metric_row["stim_reward_axis_angle_z"] = np.nan
                        else:
                            metric_row["stim_reward_axis_angle_z"] = np.nan
                    else:
                        metric_row["stim_reward_axis_angle_z"] = np.nan
    
                    all_metric_rows.append(metric_row)
                
        del raw_tensor, speed_matrix, resid_tensor, z_tensor
        gc.collect()

    # ------------------------------------------------------------
    # save scalar metrics
    # ------------------------------------------------------------
    #metrics_df = pd.DataFrame(all_metric_rows)
    #out_csv = session_metrics_dir / f"metrics_{dataset_label}_{target_file}_{area}.csv"
    #metrics_df.to_csv(out_csv, index=False)

    if len(all_pr_scaling_rows) > 0:
        pr_scaling_df = pd.concat(all_pr_scaling_rows, ignore_index=True)
    else:
        pr_scaling_df = pd.DataFrame(columns=[
            "dataset_id", "dataset_label", "group", "day", "area",
            "alignment", "epoch", "n_neurons_available", "n_trials",
            "pr_input_type", "n_neurons_subsample", "repeat", "participation_ratio"
        ])
    
    out_pr_csv = pr_scaling_dir / f"pr_scaling_{dataset_label}_{target_file}_{area}.csv"
    pr_scaling_df.to_csv(out_pr_csv, index=False)
    
    del pr_scaling_df
    gc.collect()
    # ------------------------------------------------------------
    # NEW: build one combined RDM for this session-area
    # ------------------------------------------------------------
    combined_rdm = build_combined_rdm_for_area(
        beh=beh,
        spk=spk,
        ft=ft,
        run_speed=run_speed,
        stim_pair=STIM_PAIR,
        n_early_late=100,
    )

    if combined_rdm is not None:
        combined_rdm_entry = {
            "dataset_id": target_file,
            "dataset_label": dataset_label,
            "group": group_label,
            "day": day_label,
            "area": area,
            "rdm_type": "stim_epoch_phase",
            "meta": combined_rdm["meta"],
            "rdm": combined_rdm["rdm"],
        }
    else:
        combined_rdm_entry = None

    out_pkl = session_rdms_dir / f"rdm_{dataset_label}_{target_file}_{area}.pkl"
    with open(out_pkl, "wb") as f:
        pickle.dump(combined_rdm_entry, f, protocol=pickle.HIGHEST_PROTOCOL)


    del spk, beh, ft, run_speed, metrics_df, combined_rdm, combined_rdm_entry
    gc.collect()
    return {"metrics_csv": str(out_csv), "rdm_pkl": str(out_pkl), "pr_scaling_csv": str(out_pr_csv)}

# run pass 1 ---------------------------------------------------------------------
pass1_outputs = []
for target_file, dataset_label, group_label, day_label in TARGETS:
    for area in AREAS:
        try:
            out = process_single_session_area(
                target_file=target_file,
                dataset_label=dataset_label,
                group_label=group_label,
                day_label=day_label,
                area=area,
            )
            if out is not None:
                pass1_outputs.append(out)
        except Exception as exc:
            print(f"FAILED: {dataset_label} | {target_file} | {area} -> {exc}")

print("finished pass 1")
print(f"saved {len(pass1_outputs)} session-area result blocks")

processing: sup_bef | VR2_2021_03_20_1 | rewarded | day1 | V1
N_sub:  1000
N_sub:  2000
N_sub:  3000
N_sub:  4000
N_sub:  5000
N_sub:  6000
N_sub:  7000
N_sub:  8000
N_sub:  9000
N_sub:  10000
N_sub:  11000
N_sub:  12000
N_sub:  1000
N_sub:  2000
N_sub:  3000
N_sub:  4000
N_sub:  5000
N_sub:  6000
N_sub:  7000
N_sub:  8000
N_sub:  9000
N_sub:  10000
N_sub:  11000
N_sub:  12000
N_sub:  1000
N_sub:  2000
N_sub:  3000
N_sub:  4000
N_sub:  5000
N_sub:  6000
N_sub:  7000
N_sub:  8000
N_sub:  9000
N_sub:  10000
N_sub:  11000
N_sub:  12000
N_sub:  1000
N_sub:  2000
N_sub:  3000
N_sub:  4000
N_sub:  5000
N_sub:  6000
N_sub:  7000
N_sub:  8000
N_sub:  9000
N_sub:  10000
N_sub:  11000
N_sub:  12000
FAILED: sup_bef | VR2_2021_03_20_1 | V1 -> name 'get_early_late_masks' is not defined
processing: sup_bef | VR2_2021_03_20_1 | rewarded | day1 | mHV
N_sub:  1000
N_sub:  2000
N_sub:  3000
N_sub:  4000
N_sub:  5000
N_sub:  6000
N_sub:  7000
N_sub:  8000
N_sub:  9000
N_sub:  10000
N_sub:  11000
N_sub:  

## pass 2 — reload saved metrics only

This pass works on the saved tables, not on the raw activity.

In [9]:
# ---------------------------------------------------------------------
# pass 2: reload metric tables
# ---------------------------------------------------------------------
metric_files = sorted(session_metrics_dir.glob("metrics_*.csv"))
metrics_all = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True) if metric_files else pd.DataFrame()

print("metrics_all shape:", metrics_all.shape)
display(metrics_all)

metrics_all shape: (0, 0)


""


In [10]:
# save the full tidy table
metrics_all_path = summary_dir / "all_session_area_epoch_metrics.csv"
metrics_all.to_csv(metrics_all_path, index=False)
print(metrics_all_path)

/Users/johnmadrid/GitHub/isp-unsupervised-learning/results/geometric_summary_streaming/summary/all_session_area_epoch_metrics.csv


In [11]:
pr_scaling_files = sorted(pr_scaling_dir.glob("pr_scaling_*.csv"))

pr_scaling_all = pd.concat(
    [pd.read_csv(p) for p in pr_scaling_files],
    ignore_index=True
) if pr_scaling_files else pd.DataFrame()

print(pr_scaling_all.shape)
display(pr_scaling_all.head())

(12320, 13)


,n_neurons_subsample,repeat,participation_ratio,dataset_id,dataset_label,group,day,area,alignment,epoch,n_neurons_available,n_trials,pr_input_type
0,1000,0,10.437201,TX108_2023_03_22_1,sup_aft,rewarded,day14,V1,Trial_start_time,entrance,26252,300,z
1,1000,1,10.765194,TX108_2023_03_22_1,sup_aft,rewarded,day14,V1,Trial_start_time,entrance,26252,300,z
2,1000,2,11.322212,TX108_2023_03_22_1,sup_aft,rewarded,day14,V1,Trial_start_time,entrance,26252,300,z
3,1000,3,11.394709,TX108_2023_03_22_1,sup_aft,rewarded,day14,V1,Trial_start_time,entrance,26252,300,z
4,1000,4,11.603544,TX108_2023_03_22_1,sup_aft,rewarded,day14,V1,Trial_start_time,entrance,26252,300,z


In [12]:
pr_repeat_summary = (
    pr_scaling_all
    .groupby([
        "dataset_id", "dataset_label", "group", "day", "area",
        "alignment", "epoch", "n_neurons_available",
        "pr_input_type", "n_neurons_subsample"
    ], dropna=False)["participation_ratio"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .rename(columns={
        "mean": "pr_mean_over_repeats",
        "std": "pr_std_over_repeats",
        "count": "n_repeats_used",
    })
)

pr_repeat_summary["pr_sem_over_repeats"] = (
    pr_repeat_summary["pr_std_over_repeats"] /
    np.sqrt(pr_repeat_summary["n_repeats_used"])
)

display(pr_repeat_summary.head())

,dataset_id,dataset_label,group,day,area,alignment,epoch,n_neurons_available,pr_input_type,n_neurons_subsample,pr_mean_over_repeats,pr_std_over_repeats,n_repeats_used,pr_sem_over_repeats
0,TX108_2023_03_13_1,sup_bef,rewarded,day1,V1,SoundTime,cue,32251,z,1000,18.778965,0.676363,20,0.151239
1,TX108_2023_03_13_1,sup_bef,rewarded,day1,V1,SoundTime,cue,32251,z,2000,18.760133,0.658247,20,0.147189
2,TX108_2023_03_13_1,sup_bef,rewarded,day1,V1,SoundTime,cue,32251,z,3000,19.121192,0.549966,20,0.122976
3,TX108_2023_03_13_1,sup_bef,rewarded,day1,V1,SoundTime,cue,32251,z,4000,19.149244,0.341369,20,0.076332
4,TX108_2023_03_13_1,sup_bef,rewarded,day1,V1,SoundTime,cue,32251,z,5000,19.066028,0.218828,20,0.048931


In [13]:
plot_df = pr_repeat_summary.copy()

plot_df["day_plot"] = plot_df["day"].map({
    "day1": "before",
    "day14": "after",
})
plot_df["day_plot"] = pd.Categorical(
    plot_df["day_plot"],
    categories=["before", "after"],
    ordered=True,
)

plot_df["group_plot"] = plot_df["group"].map({
    "rewarded": "supervised",
    "unrewarded": "unsupervised",
})

palette = {
    "supervised": "#1f77b4",
    "unsupervised": "#ff7f0e",
}

In [14]:
g = sns.relplot(
    data=plot_df,
    x="n_neurons_subsample",
    y="pr_mean_over_repeats",
    hue="group_plot",
    style="day_plot",
    col="epoch",
    row="area",
    kind="line",
    units="dataset_id",
    estimator=None,
    height=3.0,
    aspect=1.2,
    palette=palette,
    facet_kws={"sharex": True, "sharey": True},
)

g.set_titles(col_template="{col_name}", row_template="{row_name}")

In [15]:
plot_sub = plot_df[plot_df["alignment"] == "SoundTime"]

g = sns.relplot(
    data=plot_sub,
    x="n_neurons_subsample",
    y="pr_mean_over_repeats",
    hue="group_plot",
    style="day_plot",
    col="epoch",
    row="area",
    kind="line",
    units="dataset_id",
    estimator=None,
    height=3.0,
    aspect=1.2,
    palette=palette,
    facet_kws={"sharex": True, "sharey": True},
)

g.set_titles(col_template="{col_name}", row_template="{row_name}")

In [16]:
pr_group_summary = []
for keys, gdf in pr_repeat_summary.groupby(
    ["group", "day", "area", "alignment", "epoch", "n_neurons_subsample", "pr_input_type"],
    dropna=False
):
    row = dict(zip(
        ["group", "day", "area", "alignment", "epoch", "n_neurons_subsample", "pr_input_type"],
        keys
    ))

    vals = pd.to_numeric(gdf["pr_mean_over_repeats"], errors="coerce")
    n = vals.notna().sum()

    row["n_sessions"] = int(gdf["dataset_id"].nunique())
    row["pr_mean"] = float(np.nanmean(vals)) if n >= 1 else np.nan
    row["pr_sem"] = float(np.nanstd(vals, ddof=1) / np.sqrt(n)) if n >= 2 else np.nan

    pr_group_summary.append(row)

pr_group_summary = pd.DataFrame(pr_group_summary)
display(pr_group_summary.head())

,group,day,area,alignment,epoch,n_neurons_subsample,pr_input_type,n_sessions,pr_mean,pr_sem
0,rewarded,day1,V1,SoundTime,cue,1000,z,1,18.778965,NaN
1,rewarded,day1,V1,SoundTime,cue,2000,z,1,18.760133,NaN
2,rewarded,day1,V1,SoundTime,cue,3000,z,1,19.121192,NaN
3,rewarded,day1,V1,SoundTime,cue,4000,z,1,19.149244,NaN
4,rewarded,day1,V1,SoundTime,cue,5000,z,1,19.066028,NaN


In [17]:
plot_group = pr_group_summary.copy()

plot_group["day_plot"] = plot_group["day"].map({
    "day1": "before",
    "day14": "after",
})
plot_group["day_plot"] = pd.Categorical(
    plot_group["day_plot"],
    categories=["before", "after"],
    ordered=True,
)

plot_group["group_plot"] = plot_group["group"].map({
    "rewarded": "supervised",
    "unrewarded": "unsupervised",
})

In [18]:
g = sns.relplot(
    data=plot_group,
    x="n_neurons_subsample",
    y="pr_mean",
    hue="group_plot",
    style="day_plot",
    col="epoch",
    row="area",
    kind="line",
    height=3.0,
    aspect=1.2,
    palette=palette,
    facet_kws={"sharex": True, "sharey": True},
)

g.set_titles(col_template="{col_name}", row_template="{row_name}")

In [19]:
plot_sub = plot_group[plot_group["alignment"] == "SoundTime"].copy()

areas = list(plot_sub["area"].dropna().unique())
epochs = list(plot_sub["epoch"].dropna().unique())

fig, axes = plt.subplots(
    nrows=len(areas),
    ncols=len(epochs),
    figsize=(4.5 * len(epochs), 3.2 * len(areas)),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for i, area in enumerate(areas):
    for j, epoch in enumerate(epochs):
        ax = axes[i, j]
        cur0 = plot_sub[(plot_sub["area"] == area) & (plot_sub["epoch"] == epoch)]

        for group_plot in ["supervised", "unsupervised"]:
            for day_plot, ls in [("before", "-"), ("after", "--")]:
                cur = cur0[
                    (cur0["group_plot"] == group_plot) &
                    (cur0["day_plot"] == day_plot)
                ].sort_values("n_neurons_subsample")

                if cur.empty:
                    continue

                x = cur["n_neurons_subsample"].to_numpy()
                y = cur["pr_mean"].to_numpy()
                sem = cur["pr_sem"].to_numpy()

                ax.plot(
                    x, y,
                    linestyle=ls,
                    color=palette[group_plot],
                    label=f"{group_plot}, {day_plot}",
                )
                ax.fill_between(
                    x,
                    y - sem,
                    y + sem,
                    color=palette[group_plot],
                    alpha=0.2,
                )

        if i == 0:
            ax.set_title(epoch)

        if j == len(epochs) - 1:
            ax.text(
                1.03, 0.5, area,
                transform=ax.transAxes,
                rotation=0,
                va="center",
                ha="left",
            )

        ax.set_xlabel("number of neurons")
        ax.set_ylabel("participation ratio")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.95])

In [20]:
# ------------------------------------------------------------
# polished PR-scaling plot
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("talk")
sns.set_style("whitegrid")

# -----------------------------
# choose which summary to plot
# -----------------------------
# expected columns in pr_group_summary:
# group, day, area, alignment, epoch, n_neurons_subsample, pr_mean, pr_sem

plot_df = pr_group_summary.copy()

# -----------------------------
# display labels and ordering
# -----------------------------
plot_df["day_plot"] = plot_df["day"].map({
    "day1": "before",
    "day14": "after",
})
plot_df["day_plot"] = pd.Categorical(
    plot_df["day_plot"],
    categories=["before", "after"],
    ordered=True,
)

plot_df["group_plot"] = plot_df["group"].map({
    "rewarded": "supervised",
    "unrewarded": "unsupervised",
})

plot_df["area"] = pd.Categorical(
    plot_df["area"],
    categories=["V1", "mHV", "lHV", "aHV"],
    ordered=True,
)

# pick one alignment at a time to keep the figure readable
ALIGNMENT_TO_PLOT = "SoundTime"
plot_df = plot_df[plot_df["alignment"] == ALIGNMENT_TO_PLOT].copy()

# optional: control epoch order explicitly if desired
epoch_order = [e for e in ["cue", "late_cue", "entrance", "post_entrance"] if e in plot_df["epoch"].unique()]
if len(epoch_order) == 0:
    epoch_order = sorted(plot_df["epoch"].dropna().unique().tolist())

plot_df["epoch"] = pd.Categorical(
    plot_df["epoch"],
    categories=epoch_order,
    ordered=True,
)

palette = {
    "supervised": "#1f77b4",
    "unsupervised": "#ff7f0e",
}

line_styles = {
    "before": "-",
    "after": "--",
}

areas = [a for a in plot_df["area"].cat.categories if a in plot_df["area"].unique()]
epochs = [e for e in plot_df["epoch"].cat.categories if e in plot_df["epoch"].unique()]

fig, axes = plt.subplots(
    nrows=len(areas),
    ncols=len(epochs),
    figsize=(4.8 * len(epochs), 3.4 * len(areas)),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for i, area in enumerate(areas):
    for j, epoch in enumerate(epochs):
        ax = axes[i, j]
        sub = plot_df[(plot_df["area"] == area) & (plot_df["epoch"] == epoch)].copy()

        for group_plot in ["supervised", "unsupervised"]:
            for day_plot in ["before", "after"]:
                cur = sub[
                    (sub["group_plot"] == group_plot) &
                    (sub["day_plot"] == day_plot)
                ].sort_values("n_neurons_subsample")

                if cur.empty:
                    continue

                x = cur["n_neurons_subsample"].to_numpy()
                y = cur["pr_mean"].to_numpy()
                sem = cur["pr_sem"].to_numpy()

                valid = np.isfinite(x) & np.isfinite(y)
                if not np.any(valid):
                    continue

                x = x[valid]
                y = y[valid]
                sem = sem[valid] if len(sem) == len(valid) else np.full_like(y, np.nan, dtype=float)

                label = f"{group_plot}, {day_plot}"

                ax.plot(
                    x, y,
                    linestyle=line_styles[day_plot],
                    linewidth=2,
                    color=palette[group_plot],
                    label=label,
                )

                if np.any(np.isfinite(sem)):
                    sem_plot = np.where(np.isfinite(sem), sem, 0.0)
                    ax.fill_between(
                        x,
                        y - sem_plot,
                        y + sem_plot,
                        color=palette[group_plot],
                        alpha=0.18,
                    )

        # only top row gets epoch titles
        if i == 0:
            ax.set_title(str(epoch))

        # only rightmost column gets row labels
        if j == len(epochs) - 1:
            ax.text(
                1.04, 0.5, str(area),
                transform=ax.transAxes,
                rotation=0,
                va="center",
                ha="left",
                fontsize=12,
            )

        # x labels only on bottom row
        if i == len(areas) - 1:
            ax.set_xlabel("number of neurons")
        else:
            ax.set_xlabel("")

        # y labels only on first column
        if j == 0:
            ax.set_ylabel("participation ratio")
        else:
            ax.set_ylabel("")

        ax.ticklabel_format(style="plain", axis="x")
        sns.despine(ax=ax, top=True, right=True)

# one shared legend
legend_handles = [
    plt.Line2D([0], [0], color=palette["supervised"], lw=2, linestyle="-",  label="supervised, before"),
    plt.Line2D([0], [0], color=palette["supervised"], lw=2, linestyle="--", label="supervised, after"),
    plt.Line2D([0], [0], color=palette["unsupervised"], lw=2, linestyle="-",  label="unsupervised, before"),
    plt.Line2D([0], [0], color=palette["unsupervised"], lw=2, linestyle="--", label="unsupervised, after"),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
)

fig.suptitle(f"participation ratio scaling ({ALIGNMENT_TO_PLOT})", y=1.04)
fig.tight_layout(rect=[0, 0, 1, 0.96])

out = plot_dir / f"pr_scaling_{ALIGNMENT_TO_PLOT}.png"
fig.savefig(out, dpi=180, bbox_inches="tight")
plt.show()

print(out)

/Users/johnmadrid/GitHub/isp-unsupervised-learning/results/geometric_summary_streaming/plots/pr_scaling_SoundTime.png


In [21]:
# ------------------------------------------------------------
# summarize PR scaling:
# first average over subsampling repeats within each session
# then average over sessions within each condition
# ------------------------------------------------------------
pr_repeat_summary = (
    pr_scaling_all
    .groupby([
        "dataset_id", "dataset_label", "group", "day", "area",
        "alignment", "epoch", "n_neurons_available",
        "pr_input_type", "n_neurons_subsample"
    ], dropna=False)["participation_ratio"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .rename(columns={
        "mean": "pr_mean_over_repeats",
        "std": "pr_std_over_repeats",
        "count": "n_repeats_used"
    })
)

pr_repeat_summary["pr_sem_over_repeats"] = (
    pr_repeat_summary["pr_std_over_repeats"] /
    np.sqrt(pr_repeat_summary["n_repeats_used"])
)

pr_group_rows = []
for keys, g in pr_repeat_summary.groupby(
    ["group", "day", "area", "alignment", "epoch", "n_neurons_subsample", "pr_input_type"],
    dropna=False
):
    row = dict(zip(
        ["group", "day", "area", "alignment", "epoch", "n_neurons_subsample", "pr_input_type"],
        keys
    ))

    vals = pd.to_numeric(g["pr_mean_over_repeats"], errors="coerce")
    n = vals.notna().sum()

    row["n_sessions"] = int(g["dataset_id"].nunique())
    row["pr_mean"] = float(np.nanmean(vals)) if n >= 1 else np.nan
    row["pr_sem"] = float(np.nanstd(vals, ddof=1) / np.sqrt(n)) if n >= 2 else np.nan

    pr_group_rows.append(row)

pr_group_summary = pd.DataFrame(pr_group_rows)

display(pr_group_summary.head())

,group,day,area,alignment,epoch,n_neurons_subsample,pr_input_type,n_sessions,pr_mean,pr_sem
0,rewarded,day1,V1,SoundTime,cue,1000,z,1,18.778965,NaN
1,rewarded,day1,V1,SoundTime,cue,2000,z,1,18.760133,NaN
2,rewarded,day1,V1,SoundTime,cue,3000,z,1,19.121192,NaN
3,rewarded,day1,V1,SoundTime,cue,4000,z,1,19.149244,NaN
4,rewarded,day1,V1,SoundTime,cue,5000,z,1,19.066028,NaN


In [22]:
# ---------------------------------------------------------------------
# group summaries
# ---------------------------------------------------------------------
metric_cols = [
    "euclidean_distance_resid",
    "mahalanobis_distance_resid",
    "decoding_accuracy_z",
    "participation_ratio_z",
    "principal_angle_mean_z",
    "stim_reward_axis_angle_z",
    "mean_noise_correlation_z",
    "signal_noise_angle_z",
]

summary_rows = []
if not metrics_all.empty:
    grouped = metrics_all.groupby(["group", "day", "area", "alignment", "epoch"], dropna=False)
    for keys, g in grouped:
        row = dict(zip(["group", "day", "area", "alignment", "epoch"], keys))
        row["n_sessions"] = int(g["dataset_id"].nunique())
        for c in metric_cols:
            vals = pd.to_numeric(g[c], errors="coerce")
            row[f"{c}_mean"] = float(np.nanmean(vals)) if vals.notna().any() else np.nan
            row[f"{c}_sem"] = float(np.nanstd(vals, ddof=1) / np.sqrt(vals.notna().sum())) if vals.notna().sum() >= 2 else np.nan
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_path = summary_dir / "group_day_area_epoch_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(summary_path)
display(summary_df.head())

/Users/johnmadrid/GitHub/isp-unsupervised-learning/results/geometric_summary_streaming/summary/group_day_area_epoch_summary.csv


""


In [23]:
# ---------------------------------------------------------------------
# within-day between-group comparison table
# ---------------------------------------------------------------------
comparison_rows = []
if not summary_df.empty:
    keys = ["day", "area", "alignment", "epoch"]
    for vals, g in summary_df.groupby(keys, dropna=False):
        rew = g[g["group"] == "rewarded"]
        unrew = g[g["group"] == "unrewarded"]
        if len(rew) == 1 and len(unrew) == 1:
            row = dict(zip(keys, vals))
            for c in metric_cols:
                row[f"{c}_rewarded_mean"] = rew.iloc[0].get(f"{c}_mean", np.nan)
                row[f"{c}_unrewarded_mean"] = unrew.iloc[0].get(f"{c}_mean", np.nan)
                row[f"{c}_rewarded_minus_unrewarded"] = row[f"{c}_rewarded_mean"] - row[f"{c}_unrewarded_mean"]
            comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = summary_dir / "within_day_between_group_comparisons.csv"
comparison_df.to_csv(comparison_path, index=False)
print(comparison_path)
display(comparison_df.head())

/Users/johnmadrid/GitHub/isp-unsupervised-learning/results/geometric_summary_streaming/summary/within_day_between_group_comparisons.csv


""


In [24]:
# ---------------------------------------------------------------------
# within-group between-day comparison table
# ---------------------------------------------------------------------
day_rows = []
if not summary_df.empty:
    keys = ["group", "area", "alignment", "epoch"]
    for vals, g in summary_df.groupby(keys, dropna=False):
        d1 = g[g["day"] == "day1"]
        d14 = g[g["day"] == "day14"]
        if len(d1) == 1 and len(d14) == 1:
            row = dict(zip(keys, vals))
            for c in metric_cols:
                row[f"{c}_day14_mean"] = d14.iloc[0].get(f"{c}_mean", np.nan)
                row[f"{c}_day1_mean"] = d1.iloc[0].get(f"{c}_mean", np.nan)
                row[f"{c}_day14_minus_day1"] = row[f"{c}_day14_mean"] - row[f"{c}_day1_mean"]
            day_rows.append(row)

day_compare_df = pd.DataFrame(day_rows)
day_compare_path = summary_dir / "within_group_between_day_comparisons.csv"
day_compare_df.to_csv(day_compare_path, index=False)
print(day_compare_path)
display(day_compare_df.head())

/Users/johnmadrid/GitHub/isp-unsupervised-learning/results/geometric_summary_streaming/summary/within_group_between_day_comparisons.csv


""


In [25]:
# ---------------------------------------------------------------------
# cross-region RDM consistency
# this is computed within each session from saved RDM objects
# ---------------------------------------------------------------------
def load_session_rdms():
    all_rows = []
    for p in sorted(session_rdms_dir.glob("rdm_*.pkl")):
        with open(p, "rb") as f:
            entry = pickle.load(f)

        if entry is None:
            continue

        meta = entry["meta"]
        all_rows.append({
            "dataset_id": entry["dataset_id"],
            "dataset_label": entry["dataset_label"],
            "group": entry["group"],
            "day": entry["day"],
            "area": entry["area"],
            "rdm_type": entry["rdm_type"],
            "conditions": tuple(m["condition"] for m in meta),
            "rdm": entry["rdm"],
        })
    return all_rows

rdm_entries = load_session_rdms()
print("number of RDM entries:", len(rdm_entries))

rdm_similarity_rows = []
for (dataset_id, alignment, epoch), entries in pd.DataFrame([{
    "dataset_id": e["dataset_id"],
    "alignment": e["alignment"],
    "epoch": e["epoch"],
} for e in rdm_entries]).drop_duplicates().groupby(["dataset_id", "alignment", "epoch"]):
    sub = [e for e in rdm_entries if e["dataset_id"] == dataset_id and e["alignment"] == alignment and e["epoch"] == epoch]
    for e1, e2 in combinations(sub, 2):
        if e1["conditions"] != e2["conditions"]:
            continue
        v1 = upper_triangle_values(e1["rdm"])
        v2 = upper_triangle_values(e2["rdm"])
        if len(v1) < 2 or len(v2) < 2:
            continue
        rho = spearmanr(v1, v2).correlation
        rdm_similarity_rows.append({
            "dataset_id": e1["dataset_id"],
            "dataset_label": e1["dataset_label"],
            "group": e1["group"],
            "day": e1["day"],
            "alignment": alignment,
            "epoch": epoch,
            "area_1": e1["area"],
            "area_2": e2["area"],
            "rdm_similarity_spearman": float(rho) if rho is not None else np.nan,
        })

rdm_similarity_df = pd.DataFrame(rdm_similarity_rows)
rdm_similarity_path = summary_dir / "region_pair_rdm_similarity.csv"
rdm_similarity_df.to_csv(rdm_similarity_path, index=False)
print(rdm_similarity_path)
display(rdm_similarity_df.head())

number of RDM entries: 0


KeyError: 'dataset_id'

In [26]:
# ---------------------------------------------------------------------
# simple plotting
# one figure per metric, faceted by area and epoch
# ---------------------------------------------------------------------
sns.set_context("talk")
sns.set_style("whitegrid")

plot_metrics = [
    "euclidean_distance_resid",
    "mahalanobis_distance_resid",
    "decoding_accuracy_z",
    "participation_ratio_z",
    "principal_angle_mean_z",
    #"stim_reward_axis_angle_z",
    "mean_noise_correlation_z",
    "signal_noise_angle_z",
]

plot_df = metrics_all.copy()

plot_df["day_plot"] = plot_df["day"].map({ "day1": "before", "day14": "after",})

plot_df["day_plot"] = pd.Categorical( plot_df["day_plot"], categories=["before", "after"], ordered=True,)

tab20 = plt.get_cmap("tab20b")
stim_color = {
    'leaf1': [tab20(6),tab20(5),tab20(4)] ,  # e.g. blue-ish
    'circle1': [tab20(18),tab20(17),tab20(16)] ,   # e.g. orange-ish
}
palette = {
    "rewarded": tab20(5),
    "unrewarded": tab20(17),
}

if not metrics_all.empty:
    for metric in plot_metrics:
        g = sns.catplot(
            data=plot_df,
            x="day_plot",
            y=metric,
            hue="group",
            col="epoch",
            row="area",
            kind="point",
            dodge=True,
            errorbar="se",
            height=3.2,
            aspect=1.1,
            sharey=False,palette=palette, margin_titles=True
        )
        g.set_titles(col_template="{col_name}", row_template="{row_name}",)
        metric_label = metric.rsplit("_", 1)[0].replace("_", " ")
        g.fig.suptitle(metric_label, y=1.02)
        for ax in g.axes.flat:
            ax.set_xlabel("learning")
            ax.set_ylabel(metric_label)
            ax.title.set_rotation(0)
        for ax in g.axes[:, -1]:
            for txt in ax.texts:
                txt.set_rotation(0)
                txt.set_va("center")
                txt.set_ha("left")
        ymin = plot_df[metric].min()
        ymax = plot_df[metric].max()
        pad = 0.05 * (ymax - ymin if ymax > ymin else 1.0)
        
        for ax in g.axes.flat:
            ax.set_ylim(ymin - pad, ymax + pad)
        out = plot_dir / f"{metric}_by_day_group_area_epoch.png"
        g.savefig(out, dpi=180, bbox_inches="tight")
        plt.close(g.fig)
        print(out)

if not rdm_similarity_df.empty:
    g = sns.catplot(
        data=rdm_similarity_df,
        x="day",
        y="rdm_similarity_spearman",
        hue="group",
        col="alignment",
        row="epoch",
        kind="point",
        dodge=True,
        errorbar="se",
        height=3.2,
        aspect=1.1, sharey=True,
    )
    g.fig.suptitle("RDM consistency across region pairs", y=1.02)
    out = plot_dir / "rdm_similarity_by_day_group.png"
    g.savefig(out, dpi=180, bbox_inches="tight")
    plt.close(g.fig)
    print(out)

KeyError: 'day'

## notes

- This notebook saves the scalar metrics immediately after each block is processed.
- The group summaries and plots are built only from the saved metric tables.

### one important conceptual point
The **stimulus–reward axis angle** is only computed when the session contains a usable per-trial reward/outcome label in the behavior dictionary.  
If your behavior dict uses a different key, add it to `REWARD_LABEL_CANDIDATES`.

### another important point
The epoch-specific trial representation is currently defined as the **mean activity over frames in the epoch window**.
If you prefer another reducer, such as max, median, or concatenating frames, change `epoch_trial_matrix`.